# d-scIGM Baseline for Single-Cell RNA-seq Analysis

This notebook provides a baseline implementation using d-scIGM for interpretable single-cell RNA-seq data analysis.

## 📋 Prerequisites

Before running this baseline, please ensure you have:

1. **Followed the main installation guide** in the root directory's README to set up the conda environment
2. **Cloned the d-scIGM repository into the baseline folder:**

```bash
# Activate the scE2TM environment first
conda activate scE2TM_env

git clone https://github.com/nbnbhwyy/d-scIGM.git

In [ ]:
"""
d-scIGM Baseline for Single-Cell RNA-seq Analysis

This notebook provides a baseline implementation using d-scIGM for interpretable single-cell RNA-seq data analysis.
"""

import os
import sys
import pandas as pd
import numpy as np
import scanpy as sc
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score, silhouette_score
from sklearn import metrics
import torch
import subprocess

# Get current directory
current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")

# Define paths
data_name = 'Wang'
data_path = os.path.join(current_dir, '..', 'data', f'{data_name}_HIGHPRE.csv')
label_path = os.path.join(current_dir, '..', 'data', f'{data_name}_cell_anno.csv')
dscigm_dir = os.path.join(current_dir, 'd-scIGM', 'model')
output_dir = os.path.join(current_dir, 'output')
os.makedirs(output_dir, exist_ok=True)

print(f"Data file: {data_path}")
print(f"Label file: {label_path}")
print(f"d-scIGM directory: {dscigm_dir}")
print(f"Output directory: {output_dir}")

# Check if files exist
if not os.path.exists(data_path):
    print(f"ERROR: Data file not found at {data_path}")
    sys.exit(1)

# ============================================================
# Step 1: Prepare data and directories for d-scIGM
# ============================================================
print("\n" + "="*50)
print("STEP 1: Preparing data and directories for d-scIGM")
print("="*50)

# Create datasets directory in d-scIGM
dscigm_data_dir = os.path.join(dscigm_dir, '../datasets')
os.makedirs(dscigm_data_dir, exist_ok=True)

# Create save directory for models
dscigm_save_dir = os.path.join(dscigm_dir, '../save')
os.makedirs(dscigm_save_dir, exist_ok=True)

# Create results directory for outputs
dscigm_results_dir = os.path.join(dscigm_dir, '../results')
os.makedirs(dscigm_results_dir, exist_ok=True)

print(f"✓ Created datasets directory: {dscigm_data_dir}")
print(f"✓ Created save directory: {dscigm_save_dir}")
print(f"✓ Created results directory: {dscigm_results_dir}")

# Copy data files to d-scIGM's datasets folder
import shutil
shutil.copy2(data_path, os.path.join(dscigm_data_dir, f'{data_name}_HIGHPRE.csv'))
shutil.copy2(label_path, os.path.join(dscigm_data_dir, f'{data_name}_cell_anno.csv'))
print(f"✓ Data files copied to: {dscigm_data_dir}")

# ============================================================
# Step 2: Run d-scIGM training
# ============================================================
print("\n" + "="*50)
print("STEP 2: Running d-scIGM training")
print("="*50)

# Change to d-scIGM model directory
os.chdir(dscigm_dir)
print(f"Changed to: {os.getcwd()}")

# Run d-scIGM with minimal parameters
cmd = f"""
python main.py \
    --dataname {data_name}_HIGHPRE.csv \
    --labelname {data_name}_cell_anno.csv \
    --dataset_dir ../datasets/ \
    --output_dir ../results/ \
    --save_path ../save/
"""

print("Running command:")
print(cmd)

# Run the command
!{cmd}

# ============================================================
# Step 3: Find and copy embeddings to our output directory
# ============================================================
print("\n" + "="*50)
print("STEP 3: Locating and copying embeddings")
print("="*50)

# Go back to baseline directory
os.chdir(current_dir)

# Look for embeddings in d-scIGM's output directories
possible_dirs = [
    dscigm_results_dir,
    dscigm_save_dir,
    dscigm_dir
]

embedding_file = None
for dir_path in possible_dirs:
    if os.path.exists(dir_path):
        print(f"Searching in: {dir_path}")
        for file in os.listdir(dir_path):
            if 'embedding' in file.lower() and data_name in file:
                embedding_file = os.path.join(dir_path, file)
                print(f"✓ Found embedding file: {embedding_file}")
                break
        if embedding_file:
            break

if embedding_file:
    # Copy to our output directory with consistent naming
    target_path = os.path.join(output_dir, f'dscIGM_{data_name}_embedding.csv')
    shutil.copy2(embedding_file, target_path)
    print(f"✓ Copied to: {target_path}")
    
    # ============================================================
    # Step 4: Evaluate embeddings
    # ============================================================
    print("\n" + "="*50)
    print("STEP 4: Evaluating d-scIGM embeddings")
    print("="*50)
    
    # Load embeddings
    embeddings = pd.read_csv(target_path, index_col=0)
    print(f"Embeddings shape: {embeddings.shape}")
    
    # Load labels
    labels_df = pd.read_csv(label_path, index_col=0)
    cell_types = list(labels_df.iloc[:, 0])
    
    # Create AnnData object
    adata = sc.AnnData(embeddings.values)
    adata.obsm["X_dscIGM"] = embeddings.values
    adata.obs['cell_type'] = cell_types
    
    # Handle potential NaN values
    non_nan_indices = [i for i, x in enumerate(adata.obs['cell_type']) if pd.notna(x)]
    print(f"Cells with valid labels: {len(non_nan_indices)}/{len(adata.obs['cell_type'])}")
    
    # Construct neighborhood graph
    sc.pp.neighbors(adata, n_neighbors=15, use_rep='X')
    
    # Find optimal resolution for Louvain clustering
    max_resolution = 2
    n_steps = 20
    ari_scores = []
    
    print("Searching for optimal clustering resolution...")
    for i in range(n_steps):
        resolution = i / 10.0
        sc.tl.louvain(adata, resolution=resolution, random_state=0)
        ari = adjusted_rand_score(
            adata.obs['cell_type'][non_nan_indices],
            adata.obs['louvain'][non_nan_indices]
        )
        ari_scores.append(ari)
        print(f"Resolution {resolution:.1f}: ARI = {ari:.4f}")
    
    # Select best resolution
    best_resolution = ari_scores.index(max(ari_scores)) * 0.1
    print(f"Best resolution: {best_resolution:.1f} (ARI = {max(ari_scores):.4f})")
    
    # Final clustering
    sc.tl.louvain(adata, resolution=best_resolution, random_state=0)
    
    # Compute UMAP
    sc.tl.umap(adata)
    
    # Visualize
    sc.pl.umap(
        adata,
        color=["cell_type", "louvain"],
        wspace=0.3,
        title=['Ground Truth Cell Types', 'd-scIGM Clustering Results']
    )
    
    # Calculate metrics
    ari = adjusted_rand_score(
        adata.obs['cell_type'][non_nan_indices],
        adata.obs['louvain'][non_nan_indices]
    )
    ami = adjusted_mutual_info_score(
        adata.obs['cell_type'][non_nan_indices],
        adata.obs['louvain'][non_nan_indices]
    )
    asw = silhouette_score(
        adata.obsm["X_dscIGM"][non_nan_indices],
        adata.obs['cell_type'][non_nan_indices]
    )
    
    print("\n" + "="*50)
    print("EVALUATION METRICS")
    print("="*50)
    print(f"Adjusted Rand Index (ARI):      {ari:.4f}")
    print(f"Adjusted Mutual Information (AMI): {ami:.4f}")
    print(f"Average Silhouette Width (ASW): {asw:.4f}")
    print("="*50)
    
    # Calculate purity score
    def purity_score(y_true, y_pred):
        contingency_matrix = metrics.cluster.contingency_matrix(y_true, y_pred)
        return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)
    
    dominant_topics = np.argmax(adata.obsm["X_dscIGM"], axis=1)
    purity = purity_score(
        adata.obs['cell_type'][non_nan_indices],
        dominant_topics[non_nan_indices]
    )
    print(f"Purity Score: {purity:.4f}")
    print("="*50)
    
    # Save results
    adata.write(os.path.join(output_dir, f'dscIGM_{data_name}_results.h5ad'))
    print(f"✓ Results saved to: {os.path.join(output_dir, f'dscIGM_{data_name}_results.h5ad')}")
    
else:
    print("\n⚠️ Could not find embedding file automatically.")
    print("Please check d-scIGM output manually and copy the embedding file to:")
    print(f"  {os.path.join(output_dir, f'dscIGM_{data_name}_embedding.csv')}")

print("\n" + "="*50)
print("DONE!")
print("="*50)